# Autoencoders for anomaly detection

Courtesy of Tarje, this notebook lets you have a go at a real research task.

This goal of this notebook is for you to design an autoencoder that can detect anomalous event in high-energy physics collisions.

The dataset is a selection of processed data from the CMS experiment at CERN recorded in 2011. It contains 100k events (proton-proton collision), from which a selection is made to target W-bosons decaying to a muon and a neutrino. The variables in the dataset reflects this choice. The W-boson, the muon and the neutrino are all elementary particles.

The preprocessing is more or less complete upon loading the dataset, but you will have to construct the model (autoencoder) yourself. Some criteria are listed below to guide you towards a working model.

After you've trained the model, the performance of your model will be tested on two anomalous datasets. Your goal should be to identify the first one, and perhaps the second; the latter is quite difficult.

The pipeline is generic, feel free to modify things as you see fit, but remember to keep the model called "model" for interfacing with plotting code and similar for the dataframes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
import tensorflow as tf

#Read csv file from CERN Open Data into memory (it's not too large)
#The data contains kinematic information about the muon, as well as reconstucted information about the neutrino from conservation of momentum.
#For more details, see https://opendata.cern/record/5205

df = pd.read_csv("https://opendata.cern/record/5205/files/Wmunu.csv")

df = df.drop(["Run", "Event"], axis = 1) #Drop metadata not suitable for training
df = df.sample(frac = 1, random_state = 67) #always shuffle your data

df.head(5)

In [ ]:
#It is always useful to plot the variables/features you are working with
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

for ax, col in zip(axes, df.columns):
    ax.hist(df[col], bins=50)
    ax.set_title(col)

plt.tight_layout()
plt.show()

The ranges of the horizontal axes on the pt, MET, dxy, and iso variables hint at anomalous events inside the dataset.
There could be many reasons for this, but the goal of anomaly detection is mainly to find them, not neccesarily explain why they occur. The explaination typically involves domain knowledge.

There is no need to remove anomalous events from the dataframe, the whole idea of anomaly detection is to find them, and a good model should be able to do so. But some care should be taken as sizeable outliers can skew the mean and variances used to normalize the data.
As a cross-check for later, check if immediately anomalous events are indeed classified as anomalies (large reconstruction error).

Notice that the charge (Q) is binary and therefore a discrete variable. The typical neural network uses Z-score normalization to process its input data using a linear transformation, does such a (continous) normalization make sense for the charge?

In [ ]:
#We'll start with creating train/test/validation sets that contain 60/20/20 % of the (shuffled!) dataset
N = len(df)
df_train = df.iloc[:int(6/10 * N)]
df_val = df.iloc[int(6/10) : int(8/10 * N)]
df_test = df.iloc[int(8/10 * N):]


#Now we define the Z-score normalization, whose statistics are computed on the training set. We also define the inverse so we can easily transform back to our original data
#If you want to treat the charge differently, do so here by modifying the code.
#As some outliers are already present, the statistics computed here will be sensitive to them. Modify this if you want.
means = df_train.mean()
stds = df_train.std()

def z_score_norm(X, means, stds):
    return (X - means) / stds

def inverse_z_score_norm(Z, means, stds):
    return Z * stds + means

df_train = z_score_norm(df_train, means, stds)
df_val = z_score_norm(df_val, means, stds)
df_test = z_score_norm(df_test, means, stds)

In [ ]:
#Have a look at the transformed input distributions. This is what the model actually sees during training
#By using log-scale on the variables exhibiting outliers the distributions look much more informative
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

for ax, col in zip(axes, df.columns):
    ax.hist(df_train[col], bins=50)
    ax.set_title(col)
    if col in ["pt", "chiSq", "dxy", "iso", "MET"]:
        ax.set_yscale("log")

plt.tight_layout()
plt.show()

Now use what you know to create the model.

Be careful about activations in the last layer of the decoder block, we are after all trying to reconstruct the input data -- a softmax or relu here does not make sense.

When configuring the model using the "fit" method, remember to set both x and y to the training data.  

To automate the training process, use functionality from keras.callbacks to implement early stopping. The typical quantity to measure here is the validation loss.
When using early stopping, set the number of epochs to some larger number such that training terminates only once the validation loss stops improving. Feel free to use some other metric here for early stopping if you want.

To interface smoothly with the plotting code in following sections, call the object obtained from keras.fit for history. That is, history = model.fit(args), where "model" is your autoencoder.

In [ ]:
#Create the model. Remember to choose a suitable latent space dimension! We do not want the model to simply memorize the inputs
from keras import layers

input_shape = df.shape[1]
latent_dim = 5

model = keras.Sequential(
    [
        keras.layers.Input(shape = (input_shape,)),
        keras.layers.Dense(20, activation = "relu"),
        keras.layers.Dense(30, activation = "relu"),
        keras.layers.Dense(10, activation = "relu"),
        keras.layers.Dense(latent_dim, activation = None),
        keras.layers.Dense(10, activation = "relu"),
        keras.layers.Dense(30, activation = "relu"),
        keras.layers.Dense(20, activation = "relu"),
        keras.layers.Dense(input_shape)
    ]
)
model.summary()

In [ ]:
#Train the model using early stopping on the validation set

early_stopping = keras.callbacks.EarlyStopping(monitor = "val_loss", patience = 10, restore_best_weights = True)
epochs = 10000
batch_size = 1028

model.compile(optimizer = "adam", loss = "MeanSquaredError")
history = model.fit(
    x = df_train,
    y = df_train,
    validation_data = (df_val, df_val),
    epochs = epochs,
    batch_size = batch_size,
    callbacks = [early_stopping],
)


In [ ]:
#Have a look at the time evolution of the losses. These are stored in the history by default.

validation_loss = history.history["val_loss"]
training_loss = history.history["loss"]

plt.plot(training_loss, label = "training loss")
plt.plot(validation_loss, label = "validation loss")
plt.xlabel("Number of epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

When doing anomaly detection with autoencoders, the most relevant metric is the loss. The higher the loss, the harder the model has trying to reconstruct the input.
Consequently, events with larger losses are likely to be anomalous.

The loss is computed per batch, but we want per-event losses. Otherwise we cannot determine if singular events are anomalous. To get individual losses, we apply the model to the test set and construct a dataframe of reconstructed input variables. We can then define the mean squared error on a per-event basis

In [ ]:
#To verify the generalizability of the trained model, let us apply it to the test set.

X_reco = model.predict(df_test)
df_reco = pd.DataFrame(X_reco, columns = df_test.columns)

df_test = df_test.reset_index(drop = True) #The index has to be reset after shuffling to combine directly with the reconstructed dataframe. This is a pandas quirk

df_test["reco_error"] = np.mean((df_test - df_reco)**2, axis = 1) #Insert the MSE as a new column in the test dataframe

df_test.head(5)

In [ ]:
#Let us have a look at the reconstruction error distribution -- this is our anomaly measure
#To have sensible plots when dealing with possible large anomalies, it is often convenient to plot in terms of quantiles, such that the extremes get neglected.

def reco_error_plotter(X, quantile, log = True):
    cutoff = X["reco_error"].quantile(quantile)
    data = X["reco_error"][X["reco_error"] < cutoff]

    plt.hist(data, bins = 50)
    plt.xlabel("Reco error")
    plt.ylabel("Number of events")
    if log:
        plt.yscale("log")
    plt.title(f"Reco error distribution (up to {quantile*100:.0f}th percentile)") #percentile = 100 * quantile
    plt.show()


reco_error_plotter(df_test, 1)
reco_error_plotter(df_test, 0.99)

In [ ]:
#We can have a look at the largest outliers/anomalies to see if they obvious.
#Since we Z-score normalized our data, reconstructed variables with values much larger than are indicative of anomalous events.

biggest_anomalies = df_test.sort_values("reco_error", ascending=False).head(10)
biggest_anomalies

Validate that your model works as intended: Check if the events with the largest reconstruction error have one or more variables that are far from the typical event. Recall that the inputs are scaled using Z-score normalization, meaning values much larger than one are anomalous.



Now let us insert an anomalous sample and use the trained autoencoder to evaluate it. In fact we will create two samples. The first will be easy to detect, while the second will be harder.

Since anomalies are by definition, well -- anomalies, they should only occur in a fraction of events, say 1 in 100 or 1 in 1000. As our original sample consists of 100k events, the anomalous samples will have 100 or 1000 events each.

In [ ]:
#Create the first (easy) sample by sampling from the original dataset, normalizing and modifying the variables

df_anom = df.sample(frac = 1/1000, random_state = 65).reset_index(drop = True)
df_anom = z_score_norm(df_anom, means, stds)

df_anom["pt"] = 1.3 * df_anom["pt"]
df_anom["iso"] = 0.7 * df_anom["iso"]
df_anom["phi"] = 1.2 * df_anom["phi"]

#Predict on this new sample and compute the per-event reconstruction loss
anom_reco = model.predict(df_anom)
df_anom_reco = pd.DataFrame(anom_reco, columns = df_anom.columns)

df_anom["reco_error"] = np.mean((df_anom - df_anom_reco)**2, axis = 1)

df_anom.head(5)

To best visualize if the model classifies the anomalies correctly (assigns them a larger reconstruction loss compared to the non-anomalous data), its best to use normalized histograms or (kernel) density estimators.

We will use seaborn for the kernel density estimates (pip install seaborn if you do not already have it). You could also use matplotlib for this, but the seaborn functionality is better and more suited for this.

If you prefer the histogram approach, feel free to implement it as an exercise.

In [ ]:
!pip install seaborn
import seaborn

In [ ]:
#Create a simple kde plotter. I suggest keeping q < 1, as rogue events (extremely large reco error) always tend to be present. This is great for
#the purposes of detecting anomalies, but bad for plotting/visualization.
def kde_comparison(testset, anomaly, q=0.99):
    plt.figure()

    cutoff = testset["reco_error"].quantile(q)

    seaborn.kdeplot(testset["reco_error"].clip(upper=cutoff), label=f"Test set (< {int(q*100)}th pct)")
    seaborn.kdeplot(anomaly["reco_error"].clip(upper=cutoff), label="Anomalous sample")

    plt.xlabel("Reco error")
    plt.legend()
    plt.xlim(0, 1.2*cutoff)
    plt.show()

kde_comparison(df_test, df_anom)

There should be a clear difference between the two distributions in the above figure (although with some overlap). If not, go back and tweak your model. This is a bit contrived in the "real setting", as you don't know a priori what anomalies will look like, but this one is rather straightforward and your model should be able to find it.

For the hard sample, we will impose conditional relationships between variables. These can be hard to see when plotting the one-dimensional distributions, but may appear when inspecting two-dimensional (or higher) correlations

In [ ]:
df_corr = df.sample(frac = 1/100, random_state = 66).reset_index(drop = True)
df_corr = z_score_norm(df_corr, means, stds)

#Creating a deeper, conditional correlation
mask = df_corr["MET"] > df_corr["MET"].quantile(0.95)

df_corr.loc[mask, "pt"] = (
    df_corr.loc[mask, "pt"] + 0.2 * np.cos(df_corr.loc[mask, "phi"]) + 0.3 * df_corr.loc[mask, "phi"] + 0.05 * np.random.randn(mask.sum())
)

df_corr.loc[mask, "phiMET"] = (
    df_corr.loc[mask, "phiMET"] + 0.15 * np.sin(df_corr.loc[mask, "phi"]) + 0.25 * df_corr.loc[mask, "phi"] + 0.05 * np.random.randn(mask.sum())
)


In [ ]:
#Use the trained model to predict on this new sample, exactly the same as the previous example

corr_reco = model.predict(df_corr)
df_corr_reco = pd.DataFrame(corr_reco, columns = df_corr.columns)

df_corr["reco_error"] = np.mean((df_corr - df_corr_reco)**2, axis = 1)

In [ ]:
#Let's see if we can find this anomaly.
#This task is challenging, and you might not be able to actually detect this anomaly convincingly. If you do, congratulations!

kde_comparison(df_test, df_corr)